# 11: tabela główna: wszystkie metody na tym samym val, tą samą miarą

Bez treningu: ocena istniejących runów i metod bez uczenia na val `rs_video_x4k` (321 próbek, 13 scen spoza treningu).
Kod: `src/main_table.py` (opis metod i miar w nagłówku pliku).

* **`psnr_full`, `ssim_full`** = cały obraz, średnia po obrazach: **wspólna miara, kolumna główna**. Wcześniejsze tabele
  mieszały ją z `psnr` liczonym na obszarze widocznym w A (`train_raft.py` liczył tylko ten drugi); tu `psnr_vis` jest tylko
  dla porównania ze starymi liczbami.
* **CI95 (sceny)**: przedział ufności średniej z bootstrapu **po scenach** (val ma 13 scen po 27 próbek, próbki z jednej
  sceny nie są niezależne). Przedziały wartości bezwzględnych są szerokie; o przewadze jednej metody nad drugą mówi
  **różnica sparowana** (te same próbki) względem modelu głównego, z własnym CI i liczbą próbek, w których metoda wygrywa.
* **ms/parę**: mediana 20 przebiegów na jednej parze po rozgrzewce, sama metoda (bez wczytywania danych), na podanym
  urządzeniu; metody klasyczne na CPU (OpenCV + geometria w PyTorch).
* **kroki treningu**: epoki × (próbki / bs) danego runu (bez wcześniejszych etapów: RAFT jest pretrenowany poza projektem,
  `refine_ft` startuje z `refine`).

Nowe punkty odniesienia: **samo B, średnia A+B** (trywialne) i **klasyczny przepływ (DIS, Farnebäck z OpenCV) + nasza
geometria**: bez żadnego uczenia, bez pretreningu. Sprawdzone lokalnie 14.11: DIS + geometria v1 33,44 dB psnr_full,
czyli więcej niż każda sieć bez geometrii.

**Środowisko:** GPU (L4 albo T4) dla RAFT i sieci; całość ~15–25 min, w tym metody klasyczne na CPU ~6 min.
Wyniki zapisują się na bieżąco (`results/tabela_glowna/per_sample/<metoda>.csv`), po rozłączeniu uruchom znowu: policzone
metody są pomijane (`REDO = True` liczy od nowa).

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Ocena wszystkich metod

In [ ]:
import main_table as mt
REDO = False
OUT = P.RESULTS / 'tabela_glowna'
TAB = mt.run(DATA, OUT, runs=P.RUNS, redo=REDO, workers=WORKERS)

## 3. Tabela
Kolumna `różnica vs refine_ft` = średnia różnica psnr_full (dB) względem modelu głównego na tych samych próbkach;
`CI95 różnicy` nie zawiera 0 = różnica pewna mimo małej liczby scen.

In [ ]:
import pandas as pd
with pd.option_context('display.max_columns', 30, 'display.width', 250):
    display(TAB.drop(columns=['klucz']).set_index(['grupa', 'metoda']).round(3))
print('zapisano', OUT / 'tabela_glowna.csv')

## 4. Po ocenie
Skopiuj `tabela_glowna.csv` do `docs/` (albo daj znać, wpiszę do `experiments/README.md`). Odłącz środowisko.